# 09 — Mega-Cap Weight Contribution Decomposition

Methodology gate: distinguish the **direct value-weight effect** of mega-cap companies from the **re-ranking effect** caused by rebuilding momentum quintiles after exclusion.

Primary exclusion remains Top 10 PERMCO companies. No sector or beta neutralisation is introduced here.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel, build_market_state_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import quintile_returns, quintile_return_panel
from src.mega_cap import exclude_top_permco
from src.mega_cap_decomposition import fixed_rank_value_weighted_returns, fixed_rank_spread, decomposition_table, mega_cap_leg_diagnostics, component_hac_summary


## 1. Rebuild baseline and re-formed ExTop10 portfolios


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
clean, _ = build_clean_monthly_panel(RAW, CleaningConfig())
history = build_monthly_history_panel(RAW)
market_state = build_market_state_panel(RAW)

mom_base = attach_momentum_to_formation_universe(clean, history)
mom_base = assign_signal_quintiles(mom_base)
qret_base = quintile_returns(mom_base)
vw_base = quintile_return_panel(qret_base, return_col='VW_Return')

clean_ex10 = exclude_top_permco(clean, market_state, n=10)
mom_ex10 = attach_momentum_to_formation_universe(clean_ex10, history)
mom_ex10 = assign_signal_quintiles(mom_ex10)
qret_ex10 = quintile_returns(mom_ex10)
vw_ex10 = quintile_return_panel(qret_ex10, return_col='VW_Return')


## 2. Fixed-rank exclusion: remove mega-caps but keep baseline quintile membership


In [ ]:
fixed_qret = fixed_rank_value_weighted_returns(mom_base, market_state, n=10)
vw_fixed = fixed_rank_spread(fixed_qret)
vw_fixed.tail()


## 3. Exact additive decomposition


In [ ]:
decomp = decomposition_table(vw_base, vw_fixed, vw_ex10)
decomp[['MthCalDt','BaselineSpread','FixedRanksSpread','ReformedSpread','DirectWeightEffect','ReRankingEffect','TotalChange','AdditivityError']].tail()


In [ ]:
decomp[['DirectWeightEffect','ReRankingEffect','TotalChange']].mean()


## 4. HAC inference for decomposition components


In [ ]:
hac = component_hac_summary(decomp)
hac['DirectWeightEffect']


In [ ]:
hac['ReRankingEffect']


In [ ]:
hac['TotalChange']


## 5. Where do mega-caps sit in the baseline winner and loser legs?


In [ ]:
diag = mega_cap_leg_diagnostics(mom_base, market_state, n=10)
legs = diag[diag['SignalQuintile'].isin([1,5])].copy()
leg_summary = legs.groupby('SignalQuintile', observed=True)[['FormationWeightShare','ObservedWeightShare','ReturnContribution','TopN_SecurityCount']].agg(['mean','median'])
leg_summary


## 6. Contribution difference: winner leg minus loser leg


In [ ]:
contrib = legs.pivot(index='MthCalDt', columns='SignalQuintile', values='ReturnContribution').rename(columns={1:'Q1_MegaContribution',5:'Q5_MegaContribution'})
contrib['MegaContributionToSpread'] = contrib['Q5_MegaContribution'] - contrib['Q1_MegaContribution']
contrib.describe()


## Interpretation discipline

If TotalChange is mostly explained by DirectWeightEffect while ReRankingEffect is small, the evidence supports a portfolio-weighting mechanism rather than a change in momentum ranking quality. If re-ranking is material, the exclusion also changes the cross-sectional portfolio boundaries.
